# Summary Tables

This notebook generates summary tables from the prioritized variant list, which combines cardiovascular association results with gnomAD variant frequency data

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
prioritized_path = Path("../data/final/prioritized_variants/prioritized_variants.xlsx")

out_dir = Path("../data/final/summary_tables")
out_dir.mkdir(parents=True, exist_ok=True)

out_xlsx = out_dir / "summary_tables.xlsx"

In [3]:
prioritized_df = pd.read_excel(prioritized_path)

print(f"Prioritized variants loaded: {prioritized_df.shape[0]:,} rows, {prioritized_df.shape[1]} columns")
print(f"Unique rsIDs: {prioritized_df['rsID'].nunique():,}")
print(f"Unique genes: {prioritized_df['gene'].nunique():,}")

Prioritized variants loaded: 225,358 rows, 21 columns
Unique rsIDs: 49,050
Unique genes: 51


## Summary 1: Associations per gene and dataset

In [4]:
source_phenotype_order = [
    ("GWAS Catalog",      ""),
    ("HERMES",            "Heart failure"),
    ("CARDIoGRAMplusC4D", "Coronary artery disease"),
    ("CARDIoGRAMplusC4D", "Myocardial infarction"),
    ("FinnGen",           "Heart failure"),
    ("FinnGen",           "Coronary artery disease"),
    ("FinnGen",           "Myocardial infarction"),
]

column_labels = [
    "GWAS Catalog",
    "HERMES (HF)",
    "CARDIoGRAMplusC4D (CAD)",
    "CARDIoGRAMplusC4D (MI)",
    "FinnGen (HF)",
    "FinnGen (CAD)",
    "FinnGen (MI)",
]

all_genes = sorted(prioritized_df["gene"].dropna().unique())
summary_rows = []

for gene in all_genes:
    gene_df = prioritized_df[prioritized_df["gene"] == gene]
    row = {"Gene": gene}

    for (source, trait), label in zip(source_phenotype_order, column_labels):
        if source == "GWAS Catalog":
            count = int((gene_df["source_dataset"] == source).sum())
        else:
            count = int(
                ((gene_df["source_dataset"] == source) &
                 (gene_df["phenotype"] == trait)).sum()
            )
        row[label] = count

    row["Total"] = int(gene_df.shape[0])
    row["Unique rsIDs"] = int(gene_df["rsID"].nunique())
    summary_rows.append(row)

associations_per_gene_df = pd.DataFrame(summary_rows).set_index("Gene")

total_row = associations_per_gene_df.sum(axis=0).rename("Total").to_frame().T
total_row.index.name = "Gene"
associations_per_gene_df = pd.concat([associations_per_gene_df, total_row])

## Summary 2: Significant associations by p-value threshold

In [5]:
thresholds = {
    "p < 5e-08 (genome-wide)": 5e-8,
    "p < 1e-05 (suggestive)":  1e-5,
    "p < 0.05 (nominal)":      0.05,
}

sig_rows = []

for gene in all_genes:
    gene_df = prioritized_df[prioritized_df["gene"] == gene]
    row = {"Gene": gene}

    for label, threshold in thresholds.items():
        row[label] = int((gene_df["p_value"] < threshold).sum())

    row["Total associations"] = int(gene_df.shape[0])
    row["Unique rsIDs"] = int(gene_df["rsID"].nunique())
    sig_rows.append(row)

significant_associations_df = pd.DataFrame(sig_rows).set_index("Gene")

total_row = significant_associations_df.sum(axis=0).rename("Total").to_frame().T
total_row.index.name = "Gene"
significant_associations_df = pd.concat([significant_associations_df, total_row])

## Summary 3: Top variants per gene

In [6]:
top_variant_rows = []

for gene in all_genes:
    gene_df = prioritized_df[
        prioritized_df["gene"] == gene
    ].dropna(subset=["p_value", "rsID"])

    gene_df = gene_df.sort_values("p_value", ascending=True).drop_duplicates(subset=["rsID"])

    # Tier 1: genome-wide significant
    gws = gene_df[gene_df["p_value"] < 5e-8].copy()
    gws["significance_tier"] = "Genome-wide significant"

    # Tier 2: suggestive (not already in Tier 1)
    suggestive = gene_df[
        (gene_df["p_value"] >= 5e-8) & (gene_df["p_value"] < 1e-5)
    ].copy()
    suggestive["significance_tier"] = "Suggestive"

    # Tier 3: exploratory — only if gene has no variants in Tier 1 or Tier 2
    if len(gws) == 0 and len(suggestive) == 0:
        exploratory = gene_df.head(1).copy()
        exploratory["significance_tier"] = "Exploratory"
    else:
        exploratory = pd.DataFrame()

    top_variant_rows.append(pd.concat([gws, suggestive, exploratory], ignore_index=True))

top_variants_df = pd.concat(top_variant_rows, ignore_index=True)

## Summary 4: Functional consequence categories

In [7]:
consequence_df = (
    prioritized_df
    .drop_duplicates(subset=["rsID", "functional_consequence"])
    .groupby(["gene", "functional_consequence"])
    .size()
    .reset_index(name="variant_count")
    .pivot(
        index="gene",
        columns="functional_consequence",
        values="variant_count"
    )
    .fillna(0)
    .astype(int)
)

consequence_df.index.name = "Gene"
consequence_df.columns.name = None

total_row = consequence_df.sum(axis=0).rename("Total").to_frame().T
total_row.index.name = "Gene"
consequence_df = pd.concat([consequence_df, total_row])

In [8]:
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    associations_per_gene_df.to_excel(
        writer,
        sheet_name="1. Associations per gene",
    )
    significant_associations_df.to_excel(
        writer,
        sheet_name="2. Significant by p-value",
    )
    top_variants_df.to_excel(
        writer,
        sheet_name="3. Top variants per gene",
        index=False,
    )
    consequence_df.to_excel(
        writer,
        sheet_name="4. Functional consequences",
    )